In [2]:
import sys
import pandas as pd
import numpy as np
import torch

# from sklearn.metrics import mean_squared_error, mean_absolute_error
# from transformers import Trainer, TrainingArguments
# from transformers import PatchTSTConfig, PatchTSTForPrediction

sys.path.append('../src')

from dataset import RepositorioDados
from models.har import HarModel

# Detecta o dispositivo e a precisão usada nas operações
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
print(f"Device: {device} | Precision: {dtype}")

# Seed para resultados reproduzíveis
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

c:\Users\Lenovo\Documents\GitHub\iniciacao-cientifica\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu | Precision: torch.float32


In [3]:
# ==================== CONFIGURAÇÕES PRÉ-TREINAMENTO ====================
TIMESTAMP_COLUMN = 'timestamp'  # Coluna com timestamps
TARGET_COLUMN = ['Vol']     # Coluna a prever (volatilidade)
FEATURES = ["Vol_lag_1", "Vol_week_mean", "Vol_month_mean"]
ID_COLUMNS = []             # Sem IDs (série única)

# Tamanhos das janelas: histórico 7 dias, previsão 1 dia (dados diários)
CONTEXT_LENGTH = 512         # Janela histórica: x dias passados
FORECAST_HORIZON = 1        # Prever: x dias à frente

TRAIN_FRAC, VALID_FRAC = 0.7, 0.1  # Frações treino/validação/teste

# Hyperparâmetros do modelo
PATCH_LENGTH = 1            # Tamanho do patch (1=sem patchificação, mantém cada dia)
BATCH_SIZE = 32             # Samples por batch (reduzir se GPU memory limitada)
NUM_WORKERS = 0             # Workers para data loading (0 em Windows)
EPOCHS = 50                 # Reduzido: 50→30 (volatilidade tem ciclos curtos)
LEARNING_RATE = 1e-4        # Taxa de aprendizado

In [4]:
repo = RepositorioDados()

In [5]:
tsp, train_ds, valid_ds, test_ds = repo.executar(
    timestamp_col=TIMESTAMP_COLUMN,
    train_frac=TRAIN_FRAC,
    valid_frac=VALID_FRAC,
    context_length=CONTEXT_LENGTH,
    features=FEATURES,
    target=TARGET_COLUMN,
    id_cols=ID_COLUMNS,
    forecast_horizon=FORECAST_HORIZON
)

Carregando dados de C:\Users\Lenovo\Documents\GitHub\iniciacao-cientifica\data\raw\BTCUSDT_5m.txt
Calculando features de volatilidade
Treino: 1278 amostras | Val: 256 | Teste: 512


In [6]:
# ==================== MÉTRICAS DE AVALIAÇÃO ====================
from sklearn.metrics import mean_squared_error, mean_absolute_error

def compute_metrics(eval_pred):
    pred, labels = eval_pred
    pred = pred[0] if isinstance(pred, tuple) else pred

    # 1-step ahead
    pred_1 = pred[:, 0, 0]
    label_1 = labels[:, 0, 0]

    mse = mean_squared_error(label_1, pred_1)
    mae = mean_absolute_error(label_1, pred_1)
    rmse = np.sqrt(mse)
    mape = np.mean(
        np.abs((label_1 - pred_1) / (np.abs(label_1) + 1e-9))
    ) * 100

    return {
        "MSE": mse,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

In [7]:
import matplotlib.pyplot as plt
def evaluate_and_visualize(
    model,
    test_dataset,
    tsp,
    test_df,
    model_name="Modelo"
):
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="metric_temp",
            per_device_eval_batch_size=32,
            label_names=["future_values"]
        )
    )

    outputs = trainer.predict(test_dataset)

    preds = outputs.predictions[0] if isinstance(outputs.predictions, tuple) else outputs.predictions
    labels = outputs.label_ids

    # Desnormalização
    C = len(tsp.target_columns)
    scaler = next(iter(tsp.target_scaler_dict.values()))
    
    pred_original = scaler.inverse_transform(preds.reshape(-1, C)).reshape(preds.shape)
    labels_original = scaler.inverse_transform(labels.reshape(-1, C)).reshape(labels.shape)

    # 1-step ahead
    preds_1step = pred_original[:, 0, 0]
    labels_1step = labels_original[:, 0, 0]

    # Datas corretas
    test_dates = test_df[TIMESTAMP_COLUMN].values
    forecast_dates = test_dates[CONTEXT_LENGTH : CONTEXT_LENGTH + len(preds_1step)]

    # Métricas finais (fora do Trainer)
    metrics = {
        "MSE": mean_squared_error(labels_1step, preds_1step),
        "MAE": mean_absolute_error(labels_1step, preds_1step),
        "RMSE": np.sqrt(mean_squared_error(labels_1step, preds_1step)),
        "MAPE": np.mean(np.abs((labels_1step - preds_1step) / (np.abs(labels_1step) + 1e-9))) * 100
    }

    print(f"=== AVALIAÇÃO FINAL – {model_name} ===\n")
    for k, v in metrics.items():
        print(f"{k}: {v:.6e}")

    plt.figure(figsize=(12, 6))
    plt.plot(forecast_dates, labels_1step, label="True", color="blue")
    plt.plot(forecast_dates, preds_1step, label="Predicted", linestyle="--", color="red")
    plt.title(f"{model_name} – True vs Predicted Volatility")
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

    return forecast_dates, metrics

In [8]:
from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments    
)

In [29]:
config = PatchTSTConfig(
    do_mask_input=False,
    context_length=CONTEXT_LENGTH,
    patch_length=PATCH_LENGTH,
    num_input_channels=len(TARGET_COLUMN),
    patch_stride=PATCH_LENGTH,
    prediction_length=FORECAST_HORIZON,
    d_model=128,
    num_attention_heads=16,
    num_hidden_layers=3,
    ffn_dim=512,
    dropout=0.2,
    head_dropout=0.2,
    pooling_type=None,
    channel_attention=True, # Ativação da atenção entre canais
    scaling='std',
    loss='mse',
    pre_norm=True,
    norm_type='batchnorm',
)

model = PatchTSTForPrediction(
    config=config
    ).to(device).to(dtype)

In [30]:
train_args = TrainingArguments(
    output_dir="./patchtst_volatility",
    overwrite_output_dir=True,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    do_eval=True,
    eval_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    dataloader_num_workers=NUM_WORKERS,
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=1,
    fp16=(dtype == torch.float16),
    bf16=(dtype == torch.bfloat16),
    logging_dir="./logs_patchtst_volatility",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    label_names=["future_values"]
)

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=5,
    early_stopping_threshold=0.001
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping_callback]
)

In [31]:
print("Iniciando o treinamento do PatchTST para volatilidade...")
trainer.train()

Iniciando o treinamento do PatchTST para volatilidade...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 